# 78 — Stage A + Stage B eval (parity-locked): recall additivity + nDCG@20

Single measurement spine for the bge-v2 model. One cell builds the dev globals + the bug-fixed union top-100 once; every gate consumes them. Baselines: union+SASRec recall@100 ~0.5061 (additive +0.0387 for v1 bge_m3_ft); union->LGBM nDCG@20 ~0.1637.

Gates: C0 parity -> C1 Stage-A additivity (new-artist rescue) -> C2 fused nDCG -> C3-C6 cross-encoder / SID / composite (Phase 4/5).
Run cells 1,3,4 of nb74 are NOT needed — this notebook is self-contained.


In [ ]:
# C0a) Setup.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN']=userdata.get('HF_TOKEN'); drive.mount('/content/drive', force_remount=False)
BRANCH='recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
LOCAL='/content/recsys2026/experiments/cache'; os.makedirs(LOCAL, exist_ok=True)
for name,sub in [('retrieval_v2','recsys2026_retrieval_v2_cache'),('dense','recsys2026_dense_cache')]:
    src=f'/content/drive/MyDrive/{sub}'; dst=f'{LOCAL}/{name}'; os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src,dst)
!pip install -q --upgrade 'transformers>=4.40' 'peft>=0.11' 'sentence-transformers>=3.0' \
    'datasets' 'pandas<3.0' 'bm25s' 'lightgbm' 'FlagEmbedding>=1.3' 'tqdm'
print('setup done')


In [ ]:
# C0b) CONFIG + build the shared dev globals ONCE (parity-locked).
BGE_HUB   = 'OrRim123/recsys2026-bge-m3-music-v2-merged'   # the model nb77 produced (match RUN_NAME)
USE_STATE = False                                           # MUST match how BGE_HUB was trained (nb77 USE_STATE)
STATE_CACHE = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache'  # dev states: precompute --split test (only if USE_STATE)
CACHE_DIR='experiments/cache'; ITEM_DB='talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS=['track_name','artist_name','album_name']; TOPK=100

import sys; sys.path.insert(0,'/content/recsys2026/music-crs-baselines')
import os, json as _json, numpy as np, pandas as pd, math, torch
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.crs_baseline import build_retrieval_query
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

# Parity: BGE_HUB trained WITH a rich [STATE] block, so the DEV split needs the
# SAME StateTracker states (an empty dev [STATE] = train/serve mismatch that
# silently understates Stage-A recall). Precompute the test split here (idempotent
# per-turn cache); the coverage assert below hard-fails if it didn't take.
# Skipped entirely when USE_STATE=False (the model trained without [STATE]).
if USE_STATE:
    import subprocess
    subprocess.run([sys.executable, 'scripts/precompute_train_user_state.py',
        '--lm-type', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--cache-dir', STATE_CACHE,
        '--conv-hf', 'talkpl-ai/TalkPlayData-Challenge-Dataset', '--split', 'test'],
        check=True)

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs, turn_numbers = [],[],[],[],[],[]
goal_categories, goal_specificities, struct_queries = [],[],[]
_state_cov={'hit':0,'miss':0}
def _load_state(sid,tn):
    if not USE_STATE: return None
    p=os.path.join(STATE_CACHE,'state',f'{sid}__{int(tn)}.json')
    if os.path.exists(p):
        _state_cov['hit']+=1
        try: return _json.loads(open(p).read())
        except Exception: return {}
    _state_cov['miss']+=1
    return {}   # all-unknown (parity with a serve miss)
for sess in dev:
    df=pd.DataFrame(sess['conversations']); goal=sess.get('conversation_goal') or {}
    gtx=(goal.get('listener_goal') or '').strip(); up=sess.get('user_profile') or {}
    for _,music in df[df['role']=='music'].iterrows():
        tn=int(music['turn_number'])
        prior=df[(df['turn_number']<tn)|((df['turn_number']==tn)&(df['role']=='user'))]
        lines=[]; sm=[]
        for _,t in prior.iterrows():
            role='assistant' if t['role']=='music' else t['role']
            content=item_db.id_to_metadata(t['content']) if t['role']=='music' else t['content']
            lines.append(f'{role}: {content}')
            sm.append({'role':('assistant' if t['role']=='music' else t['role']),
                       'content':(item_db.id_to_metadata(t['content']) if t['role']=='music' else t['content'])})
        q='\n'.join(lines)
        if gtx: q=q+'\ngoal: '+gtx
        queries.append(q)
        struct_queries.append(build_retrieval_query(sm, mode='bge_m3_structured', goal_text=gtx,
                              user_profile=up, max_history_turns=6, state=_load_state(sess['session_id'],tn)))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content']); user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role']=='music')&(df['turn_number']<tn)]['content']))
        goal_categories.append(goal.get('category')); goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx=[{'history_tids':p,'user_dialog':ud} for p,ud in zip(played,user_dialogs)]
assert len(struct_queries)==len(golds)
if USE_STATE:
    _tot=_state_cov['hit']+_state_cov['miss']; _cov=_state_cov['hit']/max(1,_tot)
    print(f'C0 [STATE] dev coverage: {_state_cov["hit"]}/{_tot} ({_cov:.1%})')
    assert _cov>=0.8, (f'dev [STATE] coverage {_cov:.1%} too low — the test-split precompute '
                       f'did not populate {STATE_CACHE}/state/. Re-run it, or set USE_STATE=False '
                       f'to ablate (but then BGE_HUB must be a no-state model too). Parity break.')
print('C0 dev built:', len(golds), 'turns | USE_STATE=', USE_STATE, '| sample struct query:\n', struct_queries[0][:300])

sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                            extra_config={'use_sasrec':True,'w_sasrec':1.0})
weights=[s['weight'] for s in sas.subs]
per_sub,labels=sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
union_top=RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, TOPK)
def recall_at(c,k=TOPK): return float(np.mean([1.0 if g in x[:k] else 0.0 for x,g in zip(c,golds)]))
print('C0 union+SASRec recall@100:', round(recall_at(union_top),4))

In [ ]:
# C1) Stage-A additivity: encode BGE-v2 queries against its catalog, measure
#     union-missed-gold rescue + new-artist %. (Direct encode -> avoids the
#     factory's hardcoded embed_label.)
from sentence_transformers import SentenceTransformer
import pickle, glob
# nb77 cell 6 (embed_catalog.py) writes {cache_root}/{model.replace('/','_')}/{label}/track_embeddings.pkl
# where cache_root is the Drive retrieval_v2 cache (symlinked at experiments/cache/retrieval_v2)
# and label = RUN_NAME-merged (NO 'recsys2026-' prefix). Reconstruct that; glob-fallback for safety.
safe=BGE_HUB.replace('/','_'); label=BGE_HUB.split('/')[-1].replace('recsys2026-','')
emb_path=f'{CACHE_DIR}/retrieval_v2/dense_local/{safe}/{label}/track_embeddings.pkl'
if not os.path.exists(emb_path):
    cands=glob.glob(f'{CACHE_DIR}/retrieval_v2/dense_local/{safe}/*/track_embeddings.pkl')
    assert cands, (f'missing catalog pickle for {safe} under '
                   f'{CACHE_DIR}/retrieval_v2/dense_local — run nb77 cell 6 first')
    emb_path=cands[0]
print('catalog pickle:', emb_path)
obj=pickle.load(open(emb_path,'rb')); cat_tids=obj['track_ids']
cat_mat=obj['track_mat'].astype(np.float32); _n=np.linalg.norm(cat_mat,axis=1,keepdims=True); _n[_n==0]=1.0; cat_mat/=_n  # zero-norm guard (parity w/ build_lgbm_features)
enc=SentenceTransformer(BGE_HUB, device='cuda' if torch.cuda.is_available() else 'cpu')
q_mat=enc.encode(struct_queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
bge_top=[]
for i in range(0,len(q_mat),256):
    sims=q_mat[i:i+256]@cat_mat.T
    idx=np.argpartition(-sims,TOPK,axis=1)[:,:TOPK]
    for r,row in enumerate(idx):
        order=row[np.argsort(-sims[r,row])]
        bge_top.append([cat_tids[j] for j in order])
def _aid(t):
    a=item_db.metadata_dict.get(t,{}).get('artist_id'); return (a[0] if a else None) if isinstance(a,list) else a
um=res=resn=hyp=0
for i,g in enumerate(golds):
    inu=g in union_top[i]; inb=g in bge_top[i]
    if inu or inb: hyp+=1
    if not inu:
        um+=1
        if inb:
            res+=1; pa={_aid(t) for t in played[i]}; pa.discard(None)
            if _aid(g) not in pa: resn+=1
print('=== C1 Stage-A additivity (n=%d) ==='%len(golds))
print('  bge_v2 standalone recall@100 :', round(recall_at(bge_top),4))
print('  union+SASRec    recall@100   :', round(recall_at(union_top),4))
print('  hypothetical union recall@100:', round(hyp/len(golds),4), '(+%.4f)'%(hyp/len(golds)-recall_at(union_top)))
print('  rescued/union-missed         :', f'{res}/{um}', '(%.1f%%)'%(100*res/um if um else 0))
print('  of rescued NEW-ARTIST        :', resn, '(%.1f%%)'%(100*resn/res if res else 0))
print('  GATE: ADDITIVE' if (um and res/um>=0.05 and (resn/res if res else 0)>=0.5) else '  GATE: redundant')

In [ ]:
# C2) Stage-A fused -> LGBM rerank -> nDCG@20. Feeds bge_cos + full-catalog bge_rank
#     per candidate (parity with build_lgbm_features), and A/Bs the old reranker vs a
#     bge-retrained one (lgbm_clean_full_bge) if present. JUDGE on this DEV delta only.
import os
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
def ndcg20(ranked):
    s=0.0
    for r,g in zip(ranked,golds):
        for pos,t in enumerate(r[:20]):
            if t==g: s+=1.0/math.log2(pos+2); break
    return s/len(golds)
if USE_STATE:
    print('WARNING: the bge-LGBM feature assumes STATE-LESS queries (build_lgbm_features '
          'uses state=None). USE_STATE=True would skew bge_cos/bge_rank vs the train parquet '
          '-> keep USE_STATE=False for the bge-LGBM A/B, or add train-side state support first.')
W_BGE=0.6
fused_base=RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
fused_plus=RRF_MODEL.fuse_per_sub(per_sub+[bge_top], weights+[W_BGE], sas.k, 100)
si=labels.index('sasrec_seq'); cat_idx={t:i for i,t in enumerate(cat_tids)}
# Per-candidate bge_cos + FULL-catalog bge_rank (matches build_lgbm_features: cos =
# struct-query . track, rank = full argsort over the 47k catalog). sims computed once
# per query, reused for base+plus. Harmless to the old reranker (guards on feature presence).
def build_efpc(fb,fp):
    eb=[]; ep=[]
    for qi in range(len(q_mat)):
        sims=q_mat[qi]@cat_mat.T
        rank_arr=np.empty(len(cat_tids),dtype=np.int32); rank_arr[np.argsort(-sims)]=np.arange(1,len(cat_tids)+1)
        srank={tid:r+1 for r,tid in enumerate(per_sub[si][qi])}
        def mk(c):
            out=[]
            for t in c:
                ci=cat_idx.get(t)
                out.append({'sasrec_rank':srank.get(t,10000),
                            'bge_cos':float(sims[ci]) if ci is not None else 0.0,
                            'bge_rank':int(rank_arr[ci]) if ci is not None else 10000})
            return out
        eb.append(mk(fb[qi])); ep.append(mk(fp[qi]))
    return eb,ep
print('building per-candidate bge features (cos + full-catalog rank)...')
efpc_base,efpc_plus=build_efpc(fused_base,fused_plus)
esi=[{'played_tids':played[i],'turn_number':turn_numbers[i],'prior_track_count':len(played[i])} for i in range(len(queries))]
def score(rr,f,fe,tag):
    rk=rr.rerank(queries,f,topk=20,user_ids=user_ids,goal_categories=goal_categories,
        goal_specificities=goal_specificities,user_profiles_raw=[None]*len(queries),
        extra_features_per_candidate=fe,extra_session_info=esi)
    sc=ndcg20(rk); print(f'  {tag}: nDCG@20={round(sc,4)}'); return sc
print('=== C2 fused nDCG (DEV) ===')
print('  recall@100 union->+bge:', round(recall_at(fused_base),4),'->',round(recall_at(fused_plus),4))
rr_old=LGBM_RERANKER(ITEM_DB,['all_tracks'],CORPUS,CACHE_DIR,model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')
a=score(rr_old,fused_base,efpc_base,'union+SASRec        -> lgbm(old)')
b=score(rr_old,fused_plus,efpc_plus,'union+SASRec+bge_v2 -> lgbm(old)')
print('  delta (old reranker, bge as candidate only):', round(b-a,4))
bge_dir=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full_bge'
if os.path.exists(bge_dir):
    rr_bge=LGBM_RERANKER(ITEM_DB,['all_tracks'],CORPUS,CACHE_DIR,model_path=bge_dir)
    c=score(rr_bge,fused_plus,efpc_plus,'union+SASRec+bge_v2 -> lgbm(bge-retrained)')
    print('  delta (bge-retrained vs old baseline):', round(c-a,4), '| GATE: convert' if c-a>0 else '| flat/neg (leak? see C2-retrain note)')
else:
    print('  [lgbm_clean_full_bge not found] run the C2-retrain cell once, then re-run C2 for the A/B.')

## C3-C6 — Phase 4/5 gates (placeholders)

- C3 Stage-B cross-encoder rerank over the identical top-100: (a) LGBM, (b) LGBM+ce_score feature, (c) LGBM->top-50->CE. Gate: best >= +0.005 nDCG.
- C4 ce_score feature ablation. Gate >= +0.003.
- C5 RQ-VAE semantic-ID codes as LGBM features. Gate >= +0.002 or close.
- C6 composite read (0.5 nDCG + 0.1 Cat + 0.1 Lex + 0.3 LLM via the nb75 judge).

These land with Phase 4/5 once the cross-encoder v2 + teacher parquet + SID features exist.


## C2-retrain — LGBM with the bge channel (run once, then re-run C2)

Rebuilds the LGBM features WITH `bge_cos` + `bge_rank_inv` and retrains -> `lgbm_clean_full_bge`. Mirrors the nb74 Stage-11 clean recipe + the bge flags.

LEAK WARNING: bge-m3 was trained on the train split, and these features sample the train split, so `bge_cos`/`bge_rank` are IN-SAMPLE LEAKED in the train parquet (same class as the dropped `sasrec_rank_inv` leak: +internal val, -dev). So the LGBM's own internal val is meaningless here -- JUDGE ONLY by the nb78 C2 DEV delta above (dev queries are unseen by bge = honest). If C2-dev is flat/negative, the leak dominates -> OOF cross-fit (K bge retrains) or drop bge_cos.

In [ ]:
# C2-retrain (GPU; run ONCE, then re-run C2). See the markdown above for the LEAK protocol.
import os, pandas as pd
LGBM_DIR=f'{CACHE_DIR}/retrieval_v2/lgbm'
safe=BGE_HUB.replace('/','_'); label=BGE_HUB.split('/')[-1].replace('recsys2026-','')
BGE_CACHE=f'{CACHE_DIR}/retrieval_v2/dense_local/{safe}/{label}'   # dir with track_embeddings.pkl
TR=f'{LGBM_DIR}/lgbm_train_full_bge.parquet'; VA=f'{LGBM_DIR}/lgbm_val_full_bge.parquet'
for n,seed,out in [(15000,42,TR),(2000,7,VA)]:
    if not os.path.exists(out):
        !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
            --n-sessions {n} --topk 100 --seed {seed} \
            --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
            --bge-model {BGE_HUB} --bge-cache-dir {BGE_CACHE} \
            --cache-dir {CACHE_DIR} --out {out}
# Drop the known in-sample-leaked features (same as nb74 clean); KEEP bge_* (the test).
TRC=f'{LGBM_DIR}/lgbm_train_full_bge_clean.parquet'; VAC=f'{LGBM_DIR}/lgbm_val_full_bge_clean.parquet'
for s,d in [(TR,TRC),(VA,VAC)]:
    df=pd.read_parquet(s)
    df.drop(columns=[c for c in ['cfbpr_score','sasrec_rank_inv'] if c in df.columns]).to_parquet(d,index=False)
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} \
    --output-dir {LGBM_DIR}/lgbm_clean_full_bge
print('retrained -> lgbm_clean_full_bge ; re-run C2 for the A/B (judge the DEV delta ONLY)')